# 2.0 — Feature Engineering

Build the full feature matrix from interim data and inspect feature
distributions, coverage, and target correlations.

Loads interim data saved by `1.0-wvs-data-acquisition` (or `make all`).
Saves the processed feature matrix to `data/processed/`.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..

In [ ]:
import pickle
from pathlib import Path

import pandas as pd

from stock_data.features import build_features

## Load Interim Data

In [ ]:
INTERIM = Path("data/interim")

features_raw = pd.read_parquet(INTERIM / "features_raw.parquet")
bs_raw = pd.read_parquet(INTERIM / "bs_raw.parquet")
cf_raw = pd.read_parquet(INTERIM / "cf_raw.parquet")
annual_raw = pd.read_parquet(INTERIM / "annual_raw.parquet")
close_prices = pd.read_parquet(INTERIM / "close_prices.parquet")
macro_df = pd.read_parquet(INTERIM / "macro_df.parquet")
returns_df = pd.read_parquet(INTERIM / "returns_df.parquet")
returns_full = pd.read_parquet(INTERIM / "returns_full.parquet")

print("Loaded interim data:")
for name, df in [("features_raw", features_raw), ("bs_raw", bs_raw),
                  ("cf_raw", cf_raw), ("annual_raw", annual_raw),
                  ("close_prices", close_prices), ("macro_df", macro_df),
                  ("returns_full", returns_full)]:
    print(f"  {name:15s} {str(df.shape):>15s}")

## Build Features

In [ ]:
risk_model_df, feature_cols_all = build_features(
    features_raw, bs_raw, cf_raw, annual_raw,
    close_prices, macro_df, returns_df, returns_full,
)
print(f"Modeling rows: {risk_model_df.shape[0]}")
risk_model_df.head()

## Target Distributions

In [ ]:
print("Target distributions:")
for t in ["next_q_return", "realized_vol", "risk_adj_return"]:
    print(f"  {t}: mean={risk_model_df[t].mean():.4f}, std={risk_model_df[t].std():.4f}")
print(f"\nCorrelation matrix of targets:")
print(risk_model_df[["next_q_return", "realized_vol", "risk_adj_return"]].corr().round(3))

## Feature Summary

In [ ]:
print(f"Total feature columns: {len(feature_cols_all)}")
print(f"Sample features: {feature_cols_all[:20]}")
print(f"\nCombined dataset: {risk_model_df.shape}")

## Save Processed Data

In [ ]:
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

risk_model_df.to_parquet(PROCESSED / "risk_model_df.parquet")
with open(PROCESSED / "feature_cols.pkl", "wb") as f:
    pickle.dump(feature_cols_all, f)

print(f"Processed data saved to {PROCESSED}/")